# New Zealand Olympics Medals vs Market — Comparable Event Study Notebook

It does all four steps in one place:

1. **Filters Norway Olympic medals** from the raw Olympics medal dataset
2. **Scrapes event dates** for each unique medal event and drops medals where a specific date cannot be recovered
3. **Maps medal dates to Oslo trading days** and downloads Norwegian market data for the event window and full panel
4. **Runs the same style of analysis as the USA notebook** and exports result CSVs

## Outputs written by this notebook

- `nz_event_dates_lookup.csv`
- `nz_olympic_medals_with_event_dates.csv`
- `nz_medal_daily_counts_and_score_ALL.csv`
- `nz_medal_daily_counts_and_score_SUMMER.csv`
- `nz_medal_daily_counts_and_score_WINTER.csv`
- `nz_medals_market_panel.csv`
- `nz_results_corr_matrix.csv`
- `nz_results_diff_in_means.csv`
- `nz_results_core_regressions_long.csv`
- `nz_results_sector_regressions_long.csv`
- `nz_results_season_interaction_long.csv`
- `nz_results_lead_lag_long.csv`
- `nz_results_car_summary.csv`

## Notes on comparability

- The **Olympics side** is kept the same as the USA notebook: same medal weighting, same event-date recovery logic, same mapping to the next trading day, same regression/test families.
- The **market side** is adapted to N: Oslo exchange calendar, Norwegian market benchmark, and Norway-listed equities.
- For abnormal returns, the notebook uses a **rolling market model with OSEBX as the benchmark**. That preserves the event-study logic used in the USA analysis while using a benchmark that is appropriate for Norway.


In [43]:
%pip -q install pandas numpy requests beautifulsoup4 lxml tqdm python-dateutil rapidfuzz yfinance pandas_market_calendars statsmodels scipy

Note: you may need to restart the kernel to use updated packages.


In [44]:
import warnings
warnings.filterwarnings("ignore")

import re
import time
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import pandas_market_calendars as mcal
import statsmodels.api as sm

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from dateutil import parser as dtparser
from rapidfuzz import fuzz
from scipy import stats
from statsmodels.regression.rolling import RollingOLS

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ---------- paths ----------
RAW_CANDIDATES = [
    Path("olympic_medals.csv"),
    Path("/mnt/data/olympic_medals.csv"),
    Path("../data/olympic_medals.csv"),
]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError(
        "Could not find olympic_medals.csv. Place it in the working directory, ../data, or /mnt/data."
    )

CACHE_DIR = Path("cache_wiki_newzealand")
CACHE_DIR.mkdir(exist_ok=True)

# ---------- analysis controls ----------
FILTER_COUNTRY_3 = "NZL"
FILTER_MEDAL_TYPE = "ALL"   # ALL / GOLD / SILVER / BRONZE
YEAR_MIN, YEAR_MAX = 2000, 2024
MEDAL_WEIGHTS = {"GOLD": 3, "SILVER": 2, "BRONZE": 1}

# Broad market + major NZ-listed firms
MARKET_TICKER = "^NZ50"
TICKERS = ["^NZ50", "FPH.NZ", "SPK.NZ", "AIA.NZ", "MEL.NZ", "FBU.NZ"]

TICKER_LABELS = {
    "^NZ50": "NZ50",
    "FPH.NZ": "FPH",
    "SPK.NZ": "SPK",
    "AIA.NZ": "AIA",
    "MEL.NZ": "MEL",
    "FBU.NZ": "FBU",
}

ROLLING_WINDOW = 120
EVENT_WINDOW_DAYS = 5

# ---------- scraping session ----------
USER_AGENT = "OlympicsMarketEventStudyBot/1.0 (educational notebook)"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_REST = "https://en.wikipedia.org/api/rest_v1/page/html"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"

## 1) Load the raw medal dataset and filter to Norway

In [45]:
df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

# clean key text columns
cols_clean = [
    "discipline_title",
    "slug_game",
    "event_title",
    "event_gender",
    "medal_type",
    "country_3_letter_code",
]

for c in cols_clean:
    df[c] = df[c].astype(str).str.strip()

# filter New Zealand medals
df_nzl = df[df["country_3_letter_code"] == FILTER_COUNTRY_3].copy()

if FILTER_MEDAL_TYPE.upper() != "ALL":
    df_nzl = df_nzl[df_nzl["medal_type"] == FILTER_MEDAL_TYPE].copy()

# extract Olympic year
df_nzl["year"] = pd.to_numeric(
    df_nzl["slug_game"].str.extract(r"(\d{4})")[0],
    errors="coerce"
)

df_nzl = df_nzl.dropna(subset=["year"]).copy()
df_nzl["year"] = df_nzl["year"].astype(int)

# restrict study period
df_nzl = df_nzl[
    (df_nzl["year"] >= YEAR_MIN) &
    (df_nzl["year"] <= YEAR_MAX)
].copy()

# classify Olympics season
# Summer Olympics years are divisible by 4; Winter Olympics years in this span are not
df_nzl["season"] = np.where(df_nzl["year"] % 4 == 0, "S", "W")

# keys used to identify medals and events
MEDAL_KEY = [
    "slug_game",
    "discipline_title",
    "event_title",
    "event_gender",
    "medal_type",
    "country_3_letter_code",
]

EVENT_KEY = [
    "slug_game",
    "discipline_title",
    "event_title",
    "event_gender",
]

# remove duplicate medal rows
df_nzl_dedup_medals = df_nzl.drop_duplicates(subset=MEDAL_KEY).copy()

# create unique list of events for scraping
events_nzl = (
    df_nzl_dedup_medals[EVENT_KEY]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Filtered New Zealand rows:", df_nzl.shape)
print("Unique medal awards after dedupe:", df_nzl_dedup_medals.shape[0])
print("Unique New Zealand events to scrape:", len(events_nzl))

display(events_nzl.head(10))

Raw shape: (21697, 12)
Columns: ['discipline_title', 'slug_game', 'event_title', 'event_gender', 'medal_type', 'participant_type', 'participant_title', 'athlete_url', 'athlete_full_name', 'country_name', 'country_code', 'country_3_letter_code']


,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code
0,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/stefania-cons...,Stefania CONSTANTINI,Italy,IT,ITA
1,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/amos-mosaner,Amos MOSANER,Italy,IT,ITA
2,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/kristin-skaslien,Kristin SKASLIEN,Norway,NO,NOR
3,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/magnus-nedreg...,Magnus NEDREGOTTEN,Norway,NO,NOR
4,Curling,beijing-2022,Mixed Doubles,Mixed,BRONZE,GameTeam,Sweden,https://olympics.com/en/athletes/almida-de-val,Almida DE VAL,Sweden,SE,SWE


Filtered New Zealand rows: (93, 14)
Unique medal awards after dedupe: 74
Unique New Zealand events to scrape: 73


,slug_game,discipline_title,event_title,event_gender
0,beijing-2022,Freestyle Skiing,Men's Freeski Halfpipe,Men
1,beijing-2022,Snowboard,Women's Snowboard Big Air,Women
2,beijing-2022,Snowboard,Women's Snowboard Slopestyle,Women
3,tokyo-2020,Canoe Sprint,Women's Kayak Single 200m,Women
4,tokyo-2020,Canoe Sprint,Women's Kayak Single 500m,Women
5,tokyo-2020,Canoe Sprint,Women's Kayak Double 500m,Women
6,tokyo-2020,Boxing,Men's Heavy (81-91kg),Men
7,tokyo-2020,Rugby Sevens,Men,Men
8,tokyo-2020,Rugby Sevens,Women,Women
9,tokyo-2020,Triathlon,Men's Individual,Men


## 2) Scraping helpers for Olympic event dates

In [46]:
def slug_to_year(slug_game: str) -> Optional[int]:
    m = re.search(r"(\d{4})", str(slug_game))
    return int(m.group(1)) if m else None


def wiki_search(query: str, limit: int = 8) -> List[Dict[str, Any]]:
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "format": "json",
        "utf8": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("query", {}).get("search", [])


def choose_best_wiki_title(results: List[Dict[str, Any]], discipline: str, event: str, year: int) -> Optional[str]:
    if not results:
        return None
    target = f"{discipline} {event} {year} olympics".lower()
    scored = []
    for it in results:
        title = it.get("title", "")
        t = title.lower()
        score = 0
        if str(year) in t:
            score += 20
        if "olympic" in t:
            score += 15
        score += int(fuzz.token_set_ratio(t, target) * 0.5)
        scored.append((score, title))
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1] if scored else None


def cache_read(path: Path) -> Optional[str]:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return None


def cache_write(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def wiki_html(title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", title)[:200]
    cache_path = CACHE_DIR / f"html__{safe}.html"
    cached = cache_read(cache_path)
    if cached is not None:
        return cached

    url = f"{WIKI_REST}/{title.replace(' ', '_')}"
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    cache_write(cache_path, r.text)
    return r.text


def wiki_wikitext(title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", title)[:200]
    cache_path = CACHE_DIR / f"wikitext__{safe}.txt"
    cached = cache_read(cache_path)
    if cached is not None:
        return cached

    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "format": "json",
        "titles": title,
        "redirects": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    text = ""
    for _, page in pages.items():
        revs = page.get("revisions", [])
        if not revs:
            continue
        slots = revs[0].get("slots", {})
        text = slots.get("main", {}).get("*", "") or ""
    cache_write(cache_path, text)
    return text


def extract_infobox_dates_from_html(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    infobox = soup.find("table", class_=re.compile(r"infobox"))
    if not infobox:
        return None
    for row in infobox.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue
        key = th.get_text(" ", strip=True).lower()
        if key in ("dates", "date"):
            return td.get_text(" ", strip=True)
    return None


def extract_infobox_dates_from_wikitext(wt: str) -> Optional[str]:
    if not wt:
        return None

    for key in ["dates", "date"]:
        m = re.search(rf"\|\s*{key}\s*=\s*([^\n\|]+)", wt, flags=re.IGNORECASE)
        if m:
            val = m.group(1).strip()
            val = re.sub(r"\[\[([^\]|]+)(\|[^\]]+)?\]\]", r"\1", val)
            val = re.sub(r"\{\{[^\}]+\}\}", "", val)
            val = re.sub(r"<[^>]+>", "", val).strip()
            return val if val else None

    return None


def wiki_title_to_wikidata_qid(title: str) -> Optional[str]:
    params = {
        "action": "query",
        "prop": "pageprops",
        "titles": title,
        "format": "json",
        "redirects": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})
    for _, page in pages.items():
        qid = page.get("pageprops", {}).get("wikibase_item")
        if qid:
            return qid
    return None


def wikidata_get_event_date_end(qid: str, year: int) -> Optional[str]:
    params = {
        "action": "wbgetentities",
        "ids": qid,
        "props": "claims",
        "format": "json",
    }
    r = SESSION.get(WIKIDATA_API, params=params, timeout=30)
    r.raise_for_status()
    data = r.json().get("entities", {}).get(qid, {})
    claims = data.get("claims", {})
    for prop in ["P585", "P582", "P580"]:
        for claim in claims.get(prop, []):
            mainsnak = claim.get("mainsnak", {})
            dv = mainsnak.get("datavalue", {})
            val = dv.get("value", {})
            timestr = val.get("time")
            if not timestr:
                continue
            m = re.match(r"^[+\-]?(\d{4})-(\d{2})-(\d{2})", timestr)
            if not m:
                continue
            y, mo, da = map(int, m.groups())
            if y == year:
                return f"{y:04d}-{mo:02d}-{da:02d}"
    return None


def parse_event_date_end(datestr: str, year: int) -> Optional[str]:
    if not datestr or not year:
        return None

    s = str(datestr)
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\[[^\]]+\]", "", s)
    s = re.sub(r"\([^\)]*\)", "", s).strip()

    # try full parse directly
    try:
        dt = dtparser.parse(s, fuzzy=True, default=datetime(year, 1, 1))
        if dt.year == year:
            return dt.strftime("%Y-%m-%d")
    except Exception:
        pass

    # range forms like "29 July - 1 August 2021"
    range_match = re.search(r"(.+?)\s*-\s*(.+)", s)
    if range_match:
        rhs = range_match.group(2).strip()
        try:
            dt = dtparser.parse(rhs, fuzzy=True, default=datetime(year, 1, 1))
            if dt.year == year:
                return dt.strftime("%Y-%m-%d")
        except Exception:
            pass

    # compact range like "12-15 February 2002"
    compact = re.search(r"(\d{1,2})\s*-\s*(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", s)
    if compact:
        d2 = compact.group(2)
        mo = compact.group(3)
        yr = int(compact.group(4))
        try:
            dt = dtparser.parse(f"{d2} {mo} {yr}", fuzzy=True)
            return dt.strftime("%Y-%m-%d")
        except Exception:
            pass

    return None


def scrape_event_date(slug_game: str, discipline: str, event: str, gender: str) -> Dict[str, Any]:
    year = slug_to_year(slug_game)
    if not year:
        return {"wiki_title": None, "raw_dates": None, "event_date_end": None, "status": "no_year_in_slug"}

    queries = [
        f'"{event}" {discipline} {year} Olympics',
        f"{discipline} at the {year} Olympics {event}",
        f'"{event}" {year} Olympic {discipline}',
        f"{discipline} {event} {year} Olympic",
    ]

    wiki_title = None
    search_cache_key = re.sub(r"[^a-zA-Z0-9._-]+", "_", f"{discipline}__{event}__{year}")[:180]
    search_cache_path = CACHE_DIR / f"search__{search_cache_key}.json"
    if search_cache_path.exists():
        cached = pd.read_json(search_cache_path, typ="series").to_dict()
        wiki_title = cached.get("wiki_title")

    if not wiki_title:
        for q in queries:
            try:
                res = wiki_search(q, limit=10)
            except Exception:
                continue
            title = choose_best_wiki_title(res, discipline, event, year)
            if title:
                wiki_title = title
                break
        search_cache_path.write_text(pd.Series({"wiki_title": wiki_title}).to_json(), encoding="utf-8")

    if not wiki_title:
        return {"wiki_title": None, "raw_dates": None, "event_date_end": None, "status": "no_wiki_match"}

    try:
        html = wiki_html(wiki_title)
        raw_dates = extract_infobox_dates_from_html(html)
        if raw_dates:
            iso = parse_event_date_end(raw_dates, year)
            if iso:
                return {"wiki_title": wiki_title, "raw_dates": raw_dates, "event_date_end": iso, "status": "ok_infobox"}
            out = {"wiki_title": wiki_title, "raw_dates": raw_dates, "event_date_end": None, "status": "date_parse_failed"}
        else:
            out = {"wiki_title": wiki_title, "raw_dates": None, "event_date_end": None, "status": "no_infobox_dates"}
    except Exception as e:
        return {"wiki_title": wiki_title, "raw_dates": None, "event_date_end": None, "status": f"fetch_failed_html: {e}"}

    try:
        qid = wiki_title_to_wikidata_qid(wiki_title)
        if qid:
            wd = wikidata_get_event_date_end(qid, year)
            if wd:
                return {"wiki_title": wiki_title, "raw_dates": "Wikidata", "event_date_end": wd, "status": "ok_wikidata", "wikidata_qid": qid}
    except Exception as e:
        out["wikidata_error"] = str(e)

    try:
        wt = wiki_wikitext(wiki_title)
        wt_dates = extract_infobox_dates_from_wikitext(wt)
        if wt_dates:
            iso = parse_event_date_end(wt_dates, year)
            if iso:
                return {"wiki_title": wiki_title, "raw_dates": wt_dates, "event_date_end": iso, "status": "ok_wikitext"}
            out["wikitext_raw_dates"] = wt_dates
            out["status"] = "date_parse_failed_wikitext"
        else:
            out["wikitext_raw_dates"] = None
            if out["status"] == "no_infobox_dates":
                out["status"] = "no_dates_infobox_or_wikitext"
    except Exception as e:
        out["wikitext_error"] = str(e)

    return out


## 3) Run the Norway event-date scrape, drop unresolved events, and build daily medal counts

In [47]:
# Small sample first
sample = events_nzl.sample(min(20, len(events_nzl)), random_state=42).reset_index(drop=True)

rows = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    out = scrape_event_date(
        slug_game=r["slug_game"],
        discipline=r["discipline_title"],
        event=r["event_title"],
        gender=r["event_gender"],
    )
    rows.append({**r.to_dict(), **out})
    time.sleep(0.5)

sample_dates = pd.DataFrame(rows)
display(sample_dates.head(20))
print(sample_dates["status"].value_counts())

100%|██████████| 20/20 [00:16<00:00,  1.21it/s]


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikitext_raw_dates
0,tokyo-2020,Canoe Sprint,Women's Kayak Single 500m,Women,Chronological summary of the 2016 Summer Olympics,August 2022}},2020-08-01,ok_wikitext,NaN,NaN
1,beijing-2008,Cycling Track,Individual Pursuit men,Men,Cycling at the 2008 Summer Olympics – Men's in...,Wikidata,2008-00-00,ok_wikidata,Q1202770,NaN
2,tokyo-2020,Rowing,Women's Double Sculls,Women,Rowing at the 2020 Summer Olympics – Women's d...,23–28 July 2021,2021-07-28,ok_infobox,NaN,NaN
3,beijing-2022,Freestyle Skiing,Men's Freeski Halfpipe,Men,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN
4,rio-2016,Cycling Track,Team sprint men,Men,Cycling at the 2016 Summer Olympics – Women's ...,12 August,2016-08-12,ok_infobox,NaN,NaN
5,sydney-2000,Sailing,board Mistral women,Women,Sailing at the 1996 Summer Olympics,First race: 22 July 1996 ( 1996-07-22 ) Last r...,None,date_parse_failed_wikitext,NaN,First race: {{Start date
6,tokyo-2020,Trampoline Gymnastics,Men,Men,Gymnastics at the 2020 Summer Olympics,Wikidata,2020-00-00,ok_wikidata,Q39080753,NaN
7,rio-2016,Canoe Sprint,K-1 200m (kayak single) women,Women,Canoeing at the 2016 Summer Olympics – Women's...,15–16 August 2016,2016-08-16,ok_infobox,NaN,NaN
8,tokyo-2020,Cycling Track,Men's Omnium,Men,Cycling at the 2020 Summer Olympics – Men's om...,June 2021}},2020-06-01,ok_wikitext,NaN,NaN
9,london-2012,Cycling Track,Team Pursuit 4000m men,Men,Italy at the 1960 Summer Olympics,July 2014}},2012-07-01,ok_wikitext,NaN,NaN


status
ok_wikitext                     7
ok_infobox                      6
ok_wikidata                     4
no_dates_infobox_or_wikitext    2
date_parse_failed_wikitext      1
Name: count, dtype: int64


In [48]:
OUT_PATH = Path("nz_event_dates_lookup.csv")

if OUT_PATH.exists():
    event_dates = pd.read_csv(OUT_PATH)
    print("Loaded existing New Zealand lookup:", event_dates.shape)
else:
    event_dates = pd.DataFrame(columns=list(events_nzl.columns) + [
        "wiki_title", "raw_dates", "event_date_end", "status",
        "wikidata_qid", "wikidata_error", "wikitext_raw_dates", "wikitext_error"
    ])

done_keys = set(tuple(x) for x in event_dates[EVENT_KEY].values.tolist()) if len(event_dates) else set()

to_do = events_nzl[
    ~events_nzl.apply(lambda r: tuple(r[k] for k in EVENT_KEY) in done_keys, axis=1)
].copy()

print("Remaining New Zealand events:", len(to_do))

rows = []
for _, r in tqdm(to_do.iterrows(), total=len(to_do)):
    out = scrape_event_date(
        slug_game=r["slug_game"],
        discipline=r["discipline_title"],
        event=r["event_title"],
        gender=r["event_gender"],
    )
    rows.append({**r.to_dict(), **out})
    time.sleep(0.5)

    if len(rows) >= 100:
        event_dates = pd.concat([event_dates, pd.DataFrame(rows)], ignore_index=True)
        event_dates.to_csv(OUT_PATH, index=False)
        rows = []

if rows:
    event_dates = pd.concat([event_dates, pd.DataFrame(rows)], ignore_index=True)
    event_dates.to_csv(OUT_PATH, index=False)

print("Lookup table saved:", event_dates.shape)
display(event_dates.head())
print(event_dates["status"].value_counts())

Remaining New Zealand events: 73


100%|██████████| 73/73 [00:59<00:00,  1.23it/s]

Lookup table saved: (73, 12)


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikidata_error,wikitext_raw_dates,wikitext_error
0,beijing-2022,Freestyle Skiing,Men's Freeski Halfpipe,Men,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
1,beijing-2022,Snowboard,Women's Snowboard Big Air,Women,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
2,beijing-2022,Snowboard,Women's Snowboard Slopestyle,Women,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
3,tokyo-2020,Canoe Sprint,Women's Kayak Single 200m,Women,Canoeing at the 2020 Summer Olympics – Women's...,2 August 2021 (heats and quarterfinal) 3 Augus...,None,date_parse_failed_wikitext,NaN,NaN,2 August 2021 (heats and quarterfinal)3 August...,NaN
4,tokyo-2020,Canoe Sprint,Women's Kayak Single 500m,Women,Chronological summary of the 2016 Summer Olympics,August 2022}},2020-08-01,ok_wikitext,NaN,NaN,NaN,NaN


status
ok_wikitext                     25
ok_infobox                      24
ok_wikidata                     15
date_parse_failed_wikitext       5
no_dates_infobox_or_wikitext     4
Name: count, dtype: int64


In [49]:
missing_events = event_dates[event_dates["event_date_end"].isna()].copy()
event_dates_clean = event_dates[event_dates["event_date_end"].notna()].copy()

print("Unique events missing dates:", len(missing_events))
print("Remaining unique events after drop:", len(event_dates_clean))
if len(event_dates):
    print("Dropped %:", round(100 * len(missing_events) / len(event_dates), 2), "%")
display(missing_events.head())

# Merge dates back to New Zealand rows
df_nzl_enriched = df_nzl.merge(
    event_dates_clean[EVENT_KEY + ["event_date_end", "raw_dates", "wiki_title", "status", "wikidata_qid"]],
    on=EVENT_KEY,
    how="left",
)

df_nzl_enriched.to_csv("nz_olympic_medals_with_event_dates.csv", index=False)
print("Wrote: nz_olympic_medals_with_event_dates.csv")

# Daily aggregation exactly like USA / Norway workflow
df_nzl_enriched["event_date_end"] = pd.to_datetime(df_nzl_enriched["event_date_end"], errors="coerce")
df_medals = df_nzl_enriched.drop_duplicates(subset=MEDAL_KEY).copy()

valid_medals = {"GOLD", "SILVER", "BRONZE"}
df_medals = df_medals[df_medals["medal_type"].isin(valid_medals)].copy()
df_medals["medal_points"] = df_medals["medal_type"].map(MEDAL_WEIGHTS).astype(float)

df_medals = df_medals[
    (df_medals["event_date_end"].dt.year >= YEAR_MIN) &
    (df_medals["event_date_end"].dt.year <= YEAR_MAX)
].copy()

nz_medal_daily_all = (
    df_medals.groupby(["event_date_end", "season"])
    .agg(
        total_medals=("medal_type", "size"),
        gold=("medal_type", lambda s: (s == "GOLD").sum()),
        silver=("medal_type", lambda s: (s == "SILVER").sum()),
        bronze=("medal_type", lambda s: (s == "BRONZE").sum()),
        performance_score=("medal_points", "sum"),
    )
    .reset_index()
    .sort_values("event_date_end")
)

cols = ["event_date_end", "total_medals", "gold", "silver", "bronze", "performance_score", "season"]
nz_medal_daily_all = nz_medal_daily_all[cols]
nz_medal_daily_summer = nz_medal_daily_all[nz_medal_daily_all["season"] == "S"].copy()
nz_medal_daily_winter = nz_medal_daily_all[nz_medal_daily_all["season"] == "W"].copy()

nz_medal_daily_all.to_csv("nz_medal_daily_counts_and_score_ALL.csv", index=False)
nz_medal_daily_summer.to_csv("nz_medal_daily_counts_and_score_SUMMER.csv", index=False)
nz_medal_daily_winter.to_csv("nz_medal_daily_counts_and_score_WINTER.csv", index=False)

print("ALL:", nz_medal_daily_all.shape)
print("SUMMER:", nz_medal_daily_summer.shape)
print("WINTER:", nz_medal_daily_winter.shape)
display(nz_medal_daily_all.head())

Unique events missing dates: 9
Remaining unique events after drop: 64
Dropped %: 12.33 %


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikidata_error,wikitext_raw_dates,wikitext_error
3,tokyo-2020,Canoe Sprint,Women's Kayak Single 200m,Women,Canoeing at the 2020 Summer Olympics – Women's...,2 August 2021 (heats and quarterfinal) 3 Augus...,None,date_parse_failed_wikitext,NaN,NaN,2 August 2021 (heats and quarterfinal)3 August...,NaN
16,tokyo-2020,Athletics,Women's Shot Put,Women,Athletics at the 2020 Summer Olympics – Women'...,30 July 2021 (qualifying) 1 August 2021 (final),None,date_parse_failed_wikitext,NaN,NaN,30 July 2021 (qualifying)1 August 2021(final),NaN
30,rio-2016,Athletics,pole vault women,Women,Athletics at the 2024 Summer Olympics – Women'...,5 August 2024 (qualification) 7 August 2024 (f...,None,date_parse_failed_wikitext,NaN,NaN,{{ubl,NaN
39,rio-2016,Sailing,Laser - One Person Dinghy men,Men,Wannes Van Laer,None,None,no_dates_infobox_or_wikitext,NaN,NaN,None,NaN
41,rio-2016,Sailing,470 - Two Person Dinghy women,Women,Wang Xiaoli (sailor),None,None,no_dates_infobox_or_wikitext,NaN,NaN,None,NaN


Wrote: nz_olympic_medals_with_event_dates.csv
ALL: (40, 7)
SUMMER: (39, 7)
WINTER: (1, 7)


,event_date_end,total_medals,gold,silver,bronze,performance_score,season
0,2000-01-01,1,0,0,1,1.0,S
1,2000-06-01,1,1,0,0,3.0,S
2,2004-07-01,2,1,1,0,5.0,S
3,2004-09-01,1,1,0,0,3.0,S
4,2008-01-01,2,0,1,1,3.0,S


## 4) Map medal days to Oslo trading days and build the full trading-day panel

In [50]:
medals = pd.read_csv("nz_medal_daily_counts_and_score_ALL.csv")
medals = medals.rename(columns={"event_date_end": "medal_date"})
medals["medal_date"] = pd.to_datetime(medals["medal_date"]).dt.normalize()

# Use New Zealand exchange calendar
cal = None
for cal_name in ["XNZE", "NZE"]:
    try:
        cal = mcal.get_calendar(cal_name)
        print("Using calendar:", cal_name)
        break
    except Exception:
        pass
if cal is None:
    raise RuntimeError("Could not load a New Zealand Exchange calendar from pandas_market_calendars.")

start = medals["medal_date"].min() - pd.Timedelta(days=15)
end = medals["medal_date"].max() + pd.Timedelta(days=15)
schedule = cal.schedule(start_date=start, end_date=end)
trading_days = pd.DatetimeIndex(schedule.index).normalize()

def to_next_trading_day(d: pd.Timestamp) -> pd.Timestamp:
    d = pd.Timestamp(d).normalize()
    idx = trading_days.searchsorted(d)
    if idx >= len(trading_days):
        return pd.NaT
    return trading_days[idx]

medals["trade_date"] = medals["medal_date"].map(to_next_trading_day)
medals = medals.dropna(subset=["trade_date"]).copy()
medals["trade_date"] = pd.to_datetime(medals["trade_date"]).dt.normalize()
medals["winter_dummy"] = (medals["season"] == "W").astype(int)
medals["any_medal_dummy"] = (medals["total_medals"] > 0).astype(int)

full_schedule = cal.schedule(start_date=medals["trade_date"].min(), end_date=medals["trade_date"].max())
full_trading_days = pd.DatetimeIndex(full_schedule.index).normalize()

panel = pd.DataFrame({"trade_date": full_trading_days})
panel = panel.merge(
    medals[[
        "trade_date", "season", "gold", "silver", "bronze", "total_medals",
        "performance_score", "winter_dummy", "any_medal_dummy"
    ]],
    on="trade_date",
    how="left",
)

fill_zero = ["gold", "silver", "bronze", "total_medals", "performance_score", "winter_dummy", "any_medal_dummy"]
for c in fill_zero:
    panel[c] = panel[c].fillna(0)
panel["season"] = panel["season"].fillna("N")

print("Medal-day rows:", medals.shape)
print("Full trading-day panel:", panel.shape)
display(panel.head())

Using calendar: XNZE
Medal-day rows: (40, 10)
Full trading-day panel: (5530, 9)


,trade_date,season,gold,silver,bronze,total_medals,performance_score,winter_dummy,any_medal_dummy
0,2000-01-05,S,0.0,0.0,1.0,1.0,1.0,0.0,1.0
1,2000-01-06,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2000-01-07,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2000-01-10,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2000-01-11,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5) Download Norwegian market data and compute returns, volumes, volatility, and abnormal returns

In [51]:
market_start = panel["trade_date"].min() - pd.Timedelta(days=400)
market_end = panel["trade_date"].max() + pd.Timedelta(days=EVENT_WINDOW_DAYS + 5)

raw = yf.download(
    TICKERS,
    start=market_start.strftime("%Y-%m-%d"),
    end=(market_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    auto_adjust=False,
    progress=False,
    group_by="ticker",
)

if raw.empty:
    raise RuntimeError("Yahoo Finance returned no data for the requested New Zealand tickers.")

rows = []
for t in TICKERS:
    if isinstance(raw.columns, pd.MultiIndex):
        if t not in raw.columns.get_level_values(0):
            print(f"Skipping missing ticker: {t}")
            continue
        sub = raw[t].copy()
    else:
        sub = raw.copy()

    sub = sub.reset_index().rename(columns={"Date": "date"})
    sub.columns = [str(c) for c in sub.columns]

    if "Adj Close" in sub.columns:
        px_col = "Adj Close"
    elif "Close" in sub.columns:
        px_col = "Close"
    else:
        continue

    keep = [c for c in ["date", px_col, "Volume"] if c in sub.columns]
    sub = sub[keep].copy()
    sub = sub.rename(columns={px_col: "price", "Volume": "volume"})
    sub["date"] = pd.to_datetime(sub["date"]).dt.normalize()
    sub["ticker"] = t
    rows.append(sub)

market_long = pd.concat(rows, ignore_index=True).sort_values(["ticker", "date"]).reset_index(drop=True)

market_long["ret"] = market_long.groupby("ticker")["price"].transform(lambda s: np.log(s).diff())
market_long["lvc"] = market_long.groupby("ticker")["volume"].transform(lambda s: np.log(s.replace(0, np.nan)).diff())
market_long["vol20"] = market_long.groupby("ticker")["ret"].transform(lambda s: s.rolling(20, min_periods=10).std())

# Use NZ50 as the benchmark market series
mkt = market_long[market_long["ticker"] == MARKET_TICKER][["date", "ret"]].rename(columns={"ret": "mkt_ret"}).copy()
market_long = market_long.merge(mkt, on="date", how="left")


def compute_market_model_abret(df_t: pd.DataFrame, window: int = 120) -> pd.DataFrame:
    x = sm.add_constant(df_t[["mkt_ret"]], has_constant="add")
    y = df_t["ret"]

    valid = x.notna().all(axis=1) & y.notna()

    out = df_t.copy()
    out["exp_ret_capm"] = np.nan
    out["abret"] = np.nan

    if valid.sum() < window:
        return out

    model = RollingOLS(y[valid], x[valid], window=window)
    res = model.fit()
    params = res.params.reindex(df_t.index)

    out.loc[params.index, "exp_ret_capm"] = (
        params["const"] + params["mkt_ret"] * out.loc[params.index, "mkt_ret"]
    )
    out.loc[params.index, "abret"] = (
        out.loc[params.index, "ret"] - out.loc[params.index, "exp_ret_capm"]
    )
    return out


def compute_index_abret(df_t: pd.DataFrame, window: int = 120) -> pd.DataFrame:
    out = df_t.copy()
    out["exp_ret_capm"] = out["ret"].rolling(window, min_periods=window).mean()
    out["abret"] = out["ret"] - out["exp_ret_capm"]
    return out


parts = []
for t, g in market_long.groupby("ticker", sort=False):
    g = g.sort_values("date").copy()
    if t == MARKET_TICKER:
        g = compute_index_abret(g, window=ROLLING_WINDOW)
    else:
        g = compute_market_model_abret(g, window=ROLLING_WINDOW)
    parts.append(g)

market_long = pd.concat(parts, ignore_index=True)

display(market_long.head())

,date,price,volume,ticker,ret,lvc,vol20,mkt_ret,exp_ret_capm,abret
0,2000-01-03,0.410988,1956292.0,AIA.NZ,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,0.410988,0.0,AIA.NZ,0.000000,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,0.469211,1537852.0,AIA.NZ,0.132489,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,0.476060,326440.0,AIA.NZ,0.014492,-1.549896,NaN,NaN,NaN,NaN
4,2000-01-07,0.470923,300304.0,AIA.NZ,-0.010849,-0.083451,NaN,NaN,NaN,NaN


In [52]:
# Build a wide table for panel merge
market_long["label"] = market_long["ticker"].map(TICKER_LABELS)
market_long["ret_col"] = "ret_" + market_long["label"]
market_long["lvc_col"] = "lvc_" + market_long["label"]
market_long["absret_col"] = "absret_" + market_long["label"]
market_long["vol20_col"] = "vol20_" + market_long["label"]

ret_w = market_long.pivot(index="date", columns="ret_col", values="ret")
lvc_w = market_long.pivot(index="date", columns="lvc_col", values="lvc")
ab_w = market_long.pivot(index="date", columns="absret_col", values="abret")
vol_w = market_long.pivot(index="date", columns="vol20_col", values="vol20")

wide = pd.concat([ret_w, lvc_w, ab_w, vol_w], axis=1).reset_index().rename(columns={"date": "trade_date"})
panel = panel.merge(wide, on="trade_date", how="left")

panel.to_csv("nz_medals_market_panel.csv", index=False)
print("Wrote: nz_medals_market_panel.csv")
display(panel.head())

Wrote: nz_medals_market_panel.csv


,trade_date,season,gold,silver,bronze,total_medals,performance_score,winter_dummy,any_medal_dummy,ret_AIA,ret_FBU,ret_FPH,ret_MEL,ret_NZ50,ret_SPK,lvc_AIA,lvc_FBU,lvc_FPH,lvc_MEL,lvc_NZ50,lvc_SPK,absret_AIA,absret_FBU,absret_FPH,absret_MEL,absret_NZ50,absret_SPK,vol20_AIA,vol20_FBU,vol20_FPH,vol20_MEL,vol20_NZ50,vol20_SPK
0,2000-01-05,S,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.132489,NaN,0.113329,NaN,NaN,0.003463,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-06,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014492,NaN,0.000000,NaN,NaN,-0.017432,-1.549896,NaN,0.642156,NaN,NaN,-0.055438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-07,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.010849,NaN,0.000000,NaN,NaN,0.019734,-0.083451,NaN,-0.261861,NaN,NaN,-0.136424,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-10,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003629,NaN,0.035091,NaN,NaN,0.011429,1.734142,NaN,-0.029180,NaN,NaN,-0.242822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-11,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003617,NaN,0.006873,NaN,NaN,-0.005698,-0.538821,NaN,0.435081,NaN,NaN,0.941469,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6) Regression helpers and analysis outputs

In [53]:
def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["trade_date"] = pd.to_datetime(out["trade_date"])
    out["dow"] = out["trade_date"].dt.dayofweek
    out["month"] = out["trade_date"].dt.month

    dow_d = pd.get_dummies(out["dow"], prefix="dow", drop_first=True, dtype=int)
    mo_d = pd.get_dummies(out["month"], prefix="month", drop_first=True, dtype=int)
    out = pd.concat([out, dow_d, mo_d], axis=1)
    return out


def run_ols(df: pd.DataFrame, y: str, xvars: List[str]):
    cols = [y] + xvars
    d = df[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(d) < max(25, len(xvars) + 5):
        return None
    X = sm.add_constant(d[xvars], has_constant="add")
    model = sm.OLS(d[y], X).fit(cov_type="HC1")
    return model


def tidy_model(model, model_name: str, y_name: str) -> pd.DataFrame:
    if model is None:
        return pd.DataFrame(columns=["model", "y", "term", "coef", "std_err", "t", "p", "nobs", "r2"])
    return pd.DataFrame({
        "model": model_name,
        "y": y_name,
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t": model.tvalues.values,
        "p": model.pvalues.values,
        "nobs": model.nobs,
        "r2": model.rsquared,
    })


def diff_means(df: pd.DataFrame, col: str):
    a = df.loc[df["any_medal_dummy"] == 1, col].dropna()
    b = df.loc[df["any_medal_dummy"] == 0, col].dropna()
    if len(a) < 3 or len(b) < 3:
        return {"metric": col, "mean_medal": np.nan, "mean_nonmedal": np.nan, "diff": np.nan, "t": np.nan, "p": np.nan}
    t, p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    return {"metric": col, "mean_medal": a.mean(), "mean_nonmedal": b.mean(), "diff": a.mean() - b.mean(), "t": t, "p": p}


def compute_car(df: pd.DataFrame, ar_col: str, event_dates: pd.Series, window=(-1, 1)) -> pd.Series:
    idx_map = {d: i for i, d in enumerate(df["trade_date"])}
    vals = []
    for d in event_dates.dropna().unique():
        i = idx_map.get(pd.Timestamp(d))
        if i is None:
            continue
        lo = i + window[0]
        hi = i + window[1]
        if lo < 0 or hi >= len(df):
            continue
        vals.append(df.loc[lo:hi, ar_col].sum())
    return pd.Series(vals, name=f"CAR_{window[0]}_{window[1]}")

In [54]:
dfc = add_calendar_controls(panel)
control_cols = [c for c in dfc.columns if c.startswith("dow_") or c.startswith("month_")]

summary_cols = [
    "gold", "silver", "bronze", "total_medals", "performance_score",
    "any_medal_dummy", "winter_dummy",
    "ret_NZ50", "lvc_NZ50", "absret_NZ50", "vol20_NZ50",
]
summary_cols = [c for c in summary_cols if c in dfc.columns]

summary = dfc[summary_cols].describe().T
corr = dfc[summary_cols].corr(numeric_only=True)

summary.to_csv("nz_results_summary_stats.csv")
corr.to_csv("nz_results_corr_matrix.csv")

print("Wrote summary + correlation CSVs")
display(summary)
display(corr)

Wrote summary + correlation CSVs


,count,mean,std,min,25%,50%,75%,max
gold,5530.0,0.003797,0.069778,0.000000,0.000000,0.000000,0.000000,2.000000
silver,5530.0,0.003255,0.062995,0.000000,0.000000,0.000000,0.000000,2.000000
bronze,5530.0,0.003255,0.060056,0.000000,0.000000,0.000000,0.000000,2.000000
total_medals,5530.0,0.010307,0.137421,0.000000,0.000000,0.000000,0.000000,4.000000
performance_score,5530.0,0.021157,0.297249,0.000000,0.000000,0.000000,0.000000,9.000000
any_medal_dummy,5530.0,0.007233,0.084748,0.000000,0.000000,0.000000,0.000000,1.000000
winter_dummy,5530.0,0.000181,0.013447,0.000000,0.000000,0.000000,0.000000,1.000000
ret_NZ50,4613.0,0.000361,0.007169,-0.079468,-0.003297,0.000744,0.004316,0.069366
lvc_NZ50,4072.0,-0.002563,0.465234,-4.215522,-0.230098,0.007518,0.232060,5.712294
absret_NZ50,2450.0,0.000023,0.006964,-0.047124,-0.003692,0.000263,0.003703,0.059906


,gold,silver,bronze,total_medals,performance_score,any_medal_dummy,winter_dummy,ret_NZ50,lvc_NZ50,absret_NZ50,vol20_NZ50
gold,1.000000,0.285210,0.256010,0.750391,0.876847,0.637637,0.384773,-0.000815,-0.013874,-0.009791,-0.008780
silver,0.285210,1.000000,0.236233,0.706471,0.672440,0.605388,0.212810,0.006262,-0.014849,-0.000256,-0.021965
bronze,0.256010,0.236233,1.000000,0.675306,0.482459,0.635021,-0.000729,0.000541,-0.013363,0.014653,0.013328
total_medals,0.750391,0.706471,0.675306,1.000000,0.964332,0.878804,0.292610,0.002743,-0.019767,-0.000179,-0.008852
performance_score,0.876847,0.672440,0.482459,0.964332,1.000000,0.833943,0.361024,0.002253,-0.019287,-0.004426,-0.012886
any_medal_dummy,0.637637,0.605388,0.635021,0.878804,0.833943,1.000000,0.157555,0.010453,-0.026164,-0.001411,-0.002146
winter_dummy,0.384773,0.212810,-0.000729,0.292610,0.361024,0.157555,1.000000,0.017555,NaN,0.024978,0.000134
ret_NZ50,-0.000815,0.006262,0.000541,0.002743,0.002253,0.010453,0.017555,1.000000,0.009783,0.994356,-0.019544
lvc_NZ50,-0.013874,-0.014849,-0.013363,-0.019767,-0.019287,-0.026164,NaN,0.009783,1.000000,0.034858,-0.003716
absret_NZ50,-0.009791,-0.000256,0.014653,-0.000179,-0.004426,-0.001411,0.024978,0.994356,0.034858,1.000000,0.015460


In [55]:
metrics = [c for c in ["ret_NZ50", "lvc_NZ50", "absret_NZ50", "vol20_NZ50"] if c in dfc.columns]

diff_tbl = pd.DataFrame([diff_means(dfc, c) for c in metrics])
diff_tbl.to_csv("nz_results_diff_in_means.csv", index=False)

print("Wrote: nz_results_diff_in_means.csv")
display(diff_tbl)

Wrote: nz_results_diff_in_means.csv


,metric,mean_medal,mean_nonmedal,diff,t,p
0,ret_NZ50,0.001194,0.000354,0.000840,0.813665,0.421074
1,lvc_NZ50,-0.135202,-0.001446,-0.133756,-1.641765,0.109984
2,absret_NZ50,-0.000091,0.000024,-0.000115,-0.081376,0.936075
3,vol20_NZ50,0.006255,0.006332,-0.000078,-0.177918,0.859735


In [56]:
core_rows = []

Y_LIST = [c for c in ["ret_NZ50", "lvc_NZ50", "absret_NZ50", "vol20_NZ50"] if c in dfc.columns]

X_SETS = {
    "any_only": ["any_medal_dummy"] + control_cols,
    "score_only": ["performance_score"] + control_cols,
    "full_medal_mix": ["gold", "silver", "bronze"] + control_cols,
}

for y in Y_LIST:
    for name, xvars in X_SETS.items():
        m = run_ols(dfc, y, xvars)
        core_rows.append(tidy_model(m, f"core::{name}", y))

core = pd.concat(core_rows, ignore_index=True)
core.to_csv("nz_results_core_regressions_long.csv", index=False)

print("Wrote: nz_results_core_regressions_long.csv")
display(core.head(20))

Wrote: nz_results_core_regressions_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,core::any_only,ret_NZ50,const,0.000105,0.000399,0.262613,0.792849,4613.0,0.005756
1,core::any_only,ret_NZ50,any_medal_dummy,0.000826,0.001040,0.793798,0.427313,4613.0,0.005756
2,core::any_only,ret_NZ50,dow_1,0.000266,0.000351,0.757192,0.448935,4613.0,0.005756
3,core::any_only,ret_NZ50,dow_2,0.000457,0.000338,1.351242,0.176618,4613.0,0.005756
4,core::any_only,ret_NZ50,dow_3,-0.000331,0.000341,-0.969213,0.332439,4613.0,0.005756
5,core::any_only,ret_NZ50,dow_4,-0.000332,0.000343,-0.967088,0.333500,4613.0,0.005756
6,core::any_only,ret_NZ50,month_2,-0.000482,0.000492,-0.978471,0.327841,4613.0,0.005756
7,core::any_only,ret_NZ50,month_3,0.000632,0.000593,1.066657,0.286127,4613.0,0.005756
8,core::any_only,ret_NZ50,month_4,0.001021,0.000495,2.064671,0.038954,4613.0,0.005756
9,core::any_only,ret_NZ50,month_5,-0.000147,0.000453,-0.324402,0.745633,4613.0,0.005756


In [57]:
sec_rows = []

for label in ["FPH", "SPK", "AIA", "MEL", "FBU"]:
    for y in [f"ret_{label}", f"absret_{label}"]:
        if y not in dfc.columns:
            continue
        for name, xvars in X_SETS.items():
            m = run_ols(dfc, y, xvars)
            sec_rows.append(tidy_model(m, f"{label}::{name}", y))

sector = pd.concat(sec_rows, ignore_index=True) if sec_rows else pd.DataFrame()
sector.to_csv("nz_results_sector_regressions_long.csv", index=False)

print("Wrote: nz_results_sector_regressions_long.csv")
display(sector.head(20))

Wrote: nz_results_sector_regressions_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,FPH::any_only,ret_FPH,const,-0.000399,0.001070,-0.373219,0.708985,5522.0,0.003169
1,FPH::any_only,ret_FPH,any_medal_dummy,0.002175,0.003812,0.570529,0.568319,5522.0,0.003169
2,FPH::any_only,ret_FPH,dow_1,-0.000934,0.000769,-1.215469,0.224187,5522.0,0.003169
3,FPH::any_only,ret_FPH,dow_2,0.000322,0.000755,0.425995,0.670112,5522.0,0.003169
4,FPH::any_only,ret_FPH,dow_3,-0.000460,0.000770,-0.597028,0.550489,5522.0,0.003169
5,FPH::any_only,ret_FPH,dow_4,-0.000440,0.000741,-0.593885,0.552589,5522.0,0.003169
6,FPH::any_only,ret_FPH,month_2,-0.000098,0.001252,-0.078415,0.937498,5522.0,0.003169
7,FPH::any_only,ret_FPH,month_3,0.001994,0.001242,1.605395,0.108407,5522.0,0.003169
8,FPH::any_only,ret_FPH,month_4,0.000538,0.001195,0.449867,0.652806,5522.0,0.003169
9,FPH::any_only,ret_FPH,month_5,0.001087,0.001141,0.952502,0.340842,5522.0,0.003169


In [58]:
df_si = dfc.copy()
df_si["score_x_winter"] = df_si["performance_score"] * df_si["winter_dummy"]

si_rows = []
for y in Y_LIST:
    m = run_ols(df_si, y, ["performance_score", "winter_dummy", "score_x_winter"] + control_cols)
    si_rows.append(tidy_model(m, "season_interaction", y))

season_interaction = pd.concat(si_rows, ignore_index=True)
season_interaction.to_csv("nz_results_season_interaction_long.csv", index=False)

print("Wrote: nz_results_season_interaction_long.csv")
display(season_interaction.head(20))

Wrote: nz_results_season_interaction_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,season_interaction,ret_NZ50,const,0.000099,0.000399,0.247618,8.044303e-01,4613.0,0.005967
1,season_interaction,ret_NZ50,performance_score,-0.000113,0.000342,-0.329805,7.415470e-01,4613.0,0.005967
2,season_interaction,ret_NZ50,winter_dummy,0.000143,0.000042,3.361593,7.749435e-04,4613.0,0.005967
3,season_interaction,ret_NZ50,score_x_winter,0.001141,0.000340,3.361593,7.749435e-04,4613.0,0.005967
4,season_interaction,ret_NZ50,dow_1,0.000259,0.000351,0.736798,4.612451e-01,4613.0,0.005967
5,season_interaction,ret_NZ50,dow_2,0.000437,0.000339,1.289934,1.970736e-01,4613.0,0.005967
6,season_interaction,ret_NZ50,dow_3,-0.000335,0.000341,-0.982977,3.256188e-01,4613.0,0.005967
7,season_interaction,ret_NZ50,dow_4,-0.000342,0.000343,-0.997738,3.184064e-01,4613.0,0.005967
8,season_interaction,ret_NZ50,month_2,-0.000464,0.000493,-0.942835,3.457655e-01,4613.0,0.005967
9,season_interaction,ret_NZ50,month_3,0.000657,0.000593,1.108344,2.677132e-01,4613.0,0.005967


In [59]:
df_ll = dfc.copy()
for v in ["performance_score", "any_medal_dummy", "gold", "silver", "bronze"]:
    if v in df_ll.columns:
        df_ll[f"{v}_lag1"] = df_ll[v].shift(1)
        df_ll[f"{v}_lead1"] = df_ll[v].shift(-1)
ll_rows = []

for y in Y_LIST:
    xvars = [c for c in ["performance_score_lag1", "performance_score", "performance_score_lead1"] if c in df_ll.columns] + control_cols
    m = run_ols(df_ll, y, xvars)
    ll_rows.append(tidy_model(m, "lead_lag_score", y))

lead_lag = pd.concat(ll_rows, ignore_index=True)
lead_lag.to_csv("nz_results_lead_lag_long.csv", index=False)

print("Wrote: nz_results_lead_lag_long.csv")
display(lead_lag.head(20))


Wrote: nz_results_lead_lag_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,lead_lag_score,ret_NZ50,const,0.000104,0.000399,0.260753,7.942828e-01,4612.0,0.006115
1,lead_lag_score,ret_NZ50,performance_score_lag1,0.000062,0.000341,0.180556,8.567162e-01,4612.0,0.006115
2,lead_lag_score,ret_NZ50,performance_score,-0.000223,0.000368,-0.605999,5.445153e-01,4612.0,0.006115
3,lead_lag_score,ret_NZ50,performance_score_lead1,0.000478,0.000240,1.989927,4.659900e-02,4612.0,0.006115
4,lead_lag_score,ret_NZ50,dow_1,0.000260,0.000351,0.740444,4.590304e-01,4612.0,0.006115
5,lead_lag_score,ret_NZ50,dow_2,0.000430,0.000339,1.269054,2.044219e-01,4612.0,0.006115
6,lead_lag_score,ret_NZ50,dow_3,-0.000337,0.000342,-0.985999,3.241338e-01,4612.0,0.006115
7,lead_lag_score,ret_NZ50,dow_4,-0.000363,0.000344,-1.057731,2.901783e-01,4612.0,0.006115
8,lead_lag_score,ret_NZ50,month_2,-0.000474,0.000493,-0.961303,3.364000e-01,4612.0,0.006115
9,lead_lag_score,ret_NZ50,month_3,0.000651,0.000593,1.097729,2.723229e-01,4612.0,0.006115


In [60]:
car_rows = []
event_dates = dfc.loc[dfc["any_medal_dummy"] == 1, "trade_date"]

for label in ["NZ50", "FPH", "SPK", "AIA", "MEL", "FBU"]:
    ar_col = f"absret_{label}"
    if ar_col not in dfc.columns:
        continue

    for win in [(-1, 1), (0, 1), (0, 3), (0, 5)]:
        s = compute_car(dfc.reset_index(drop=True), ar_col, event_dates, window=win)
        car_rows.append({
            "ticker": label,
            "window": f"{win[0]} to {win[1]}",
            "n_events": int(s.notna().sum()),
            "mean_car": s.mean(),
            "median_car": s.median(),
            "std_car": s.std(),
            "t_stat_vs_0": (
                s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))
            ) if len(s) > 1 and s.std(ddof=1) > 0 else np.nan,
        })

car = pd.DataFrame(car_rows)
car.to_csv("nz_results_car_summary.csv", index=False)

print("Wrote: nz_results_car_summary.csv")
display(car)

Wrote: nz_results_car_summary.csv


,ticker,window,n_events,mean_car,median_car,std_car,t_stat_vs_0
0,NZ50,-1 to 1,37,-0.000087,0.000000,0.003918,-0.135116
1,NZ50,0 to 1,38,-0.000070,0.000000,0.004462,-0.096041
2,NZ50,0 to 3,38,-0.001421,0.000000,0.007689,-1.139045
3,NZ50,0 to 5,38,-0.002365,0.000000,0.010163,-1.434540
4,FPH,-1 to 1,37,0.000006,0.002345,0.025127,0.001502
5,FPH,0 to 1,38,0.001591,0.002074,0.020201,0.485432
6,FPH,0 to 3,38,0.003099,0.002471,0.038591,0.495075
7,FPH,0 to 5,38,-0.005036,-0.005779,0.048628,-0.638439
8,SPK,-1 to 1,37,-0.004925,-0.008362,0.017924,-1.671280
9,SPK,0 to 1,38,-0.001365,-0.000342,0.015205,-0.553302
